# 06 · Questions and qrels

Plan section 2. Validates the question files of the active experiment and derives its qrels for both indexes from character spans. The experiment is set in `experiments.toml` (`active`, or `CSRAG_EXPERIMENT`); the first cell prints it. The qrels go to `results/<experiment>/qrels.jsonl`, unless the experiment names another path (`run1` keeps `data/eval/qrels.jsonl`).

Grades:

* **unit / window level** (per index): grade 2 = the unit or window contains the whole quote span; grade 1 = another unit or window of a gold section. If no unit of the gold section holds the whole span (a quote cut by a base-unit split), the units overlapping it get grade 2 and are flagged `partial`.
* **section level** (same for both indexes): grade 2 = a gold section; grade 1 = a sibling in the same group.
* **group level**: grade 2 = the gold group.

Only answerable questions get qrels. Unanswerable ones run through every retriever but are never scored.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
from csrag import experiment
EXP = experiment.active()  # `active` in experiments.toml, or the CSRAG_EXPERIMENT environment variable
RESULTS = ROOT / EXP.dir   # results/<experiment name>/: every result of this experiment
print(EXP.summary())

experiment 'run2' -> results/run2/  (Question set 1 + question set 2 (paraphrase, section_id, twin_topic, multi_section))
   100  data/eval/questions.jsonl
   177  data/eval/set2/paraphrase.jsonl
    50  data/eval/set2/section_id.jsonl
    60  data/eval/set2/twin_topic.jsonl
    90  data/eval/set2/multi_section.jsonl
   477  questions, 469 answerable | qrels: results/run2/qrels.jsonl


In [2]:
import json
import pandas as pd
from csrag.corpus import Corpus
from csrag import evaluate as ev

corpus = Corpus(DATA)
assert corpus.has_windows, "run notebook 04 first"
questions, problems = ev.load_questions(EXP.questions, corpus)
for k, v in problems.items():
    print(f"{k:30} {len(v):3}  {v[:5]}")
assert not any(v for k, v in problems.items() if k != "quote_ambiguous"), "question validation failed"
print(len(questions), "questions;", sum(q["answerable"] for q in questions), "answerable")

duplicate_ids                    0  []
unknown_section                  0  []
quote_not_verbatim               0  []
quote_not_in_normalised_body     0  []
quote_ambiguous                  0  []
answerable_without_gold          0  []
unanswerable_with_gold           0  []
477 questions; 469 answerable


## Distribution

In [3]:
qf = pd.DataFrame([{k: q.get(k) for k in ("question_id", "question_type", "engine_type", "difficulty", "gold_kind",
                                          "mentions_section_id", "answerable")} for q in questions])
qf.insert(1, "family", [ev.family_of(q) for q in questions])  # question set 1 has no `family` field: "set1"
def length_bucket(q):
    if not q["answerable"]:
        return "unanswerable"
    n = corpus.n_tokens(corpus.sections[q["_gold"][0]["section_id"]]["body"])
    return "≤500" if n <= 500 else ("501–1500" if n <= 1500 else ">1500")
qf["length_bucket"] = [length_bucket(q) for q in questions]
for col in ("family", "question_type", "engine_type", "difficulty", "gold_kind", "mentions_section_id", "length_bucket"):
    display(qf[qf["answerable"]][col].value_counts(dropna=False).rename(col).to_frame().T)

family,paraphrase,set1,multi_section,twin_topic,section_id
family,177,92,90,60,50


question_type,factual,numeric,list,procedural,definition,summary,rationale
question_type,208,111,62,57,15,10,6


engine_type,turbine,all,piston
engine_type,228,125,116


difficulty,medium,hard,easy
difficulty,253,110,106


gold_kind,cs,amc,amc_general,appendix
gold_kind,324,141,3,1


mentions_section_id,False,True
mentions_section_id,401,68


length_bucket,≤500,501–1500,>1500
length_bucket,317,112,40


## Qrels for both indexes

In [4]:
qrels = ev.build_qrels(questions, corpus)
check = ev.check_qrels(qrels, questions, corpus)
for k, v in check.items():
    print(f"{k:22} {len(v):3}  {v[:5]}")
assert not check["orphan_ids"] and not check["missing"], "qrels have orphans or gaps"
ev.write_qrels(qrels, EXP.qrels_path)
print(len(qrels), "qrels ->", EXP.qrels_path)
qr = pd.DataFrame(qrels)
qr.groupby(["level", qr["index"].fillna("-"), "grade"]).size().unstack("grade")

orphan_ids               0  []
missing                  0  []
partial_unit_qrels       0  []
3693 qrels -> results/run2/qrels.jsonl


grade               1      2
level   index               
group   -         NaN  529.0
section -       562.0  559.0
unit    units   199.0  559.0
        win500  701.0  584.0

Relevant (grade-2) units per question: with a 50-token overlap a short quote can lie in two consecutive windows, so a window question can have two grade-2 windows.

In [5]:
g2 = qr[(qr["level"] == "unit") & (qr["grade"] == 2)].groupby(["index", "question_id"]).size()
g2.groupby("index").describe()[["mean", "min", "max"]]

,mean,min,max
index,,,
units,1.191898,1.0,2.0
win500,1.245203,1.0,4.0


## One example: a gold quote and the units that contain it

In [6]:
q = next(q for q in questions if q["answerable"] and corpus.sections[q["_gold"][0]["section_id"]]["kind"] == "amc"
         and len(corpus.units_of(q["_gold"][0]["section_id"], "win500")) > 2)
g = q["_gold"][0]
print(q["question_id"], q["question"], "\n", g)
for index in corpus.indexes():
    for u in corpus.units_of(g["section_id"], index):
        inside = u["char_start"] <= g["char_start"] and u["char_end"] >= g["char_end"]
        print(f"  {index:6} {u['unit_id']:24} [{u['char_start']:6}, {u['char_end']:6})  {'contains the quote' if inside else ''}")

q006 For a turbine engine, which installation assumptions should be declared regarding the oil system? 
 {'section_id': 'AMC E 30', 'group_id': 'CS-E 30', 'role': 'primary', 'char_start': 2206, 'char_end': 2378}
  units  AMC E 30                 [     0,   3920)  contains the quote
  win500 AMC E 30#w1              [     0,   1774)  
  win500 AMC E 30#w2              [  1622,   3354)  contains the quote
  win500 AMC E 30#w3              [  3162,   3920)  
